In [1]:
import onnxruntime as ort

session = ort.InferenceSession(
    "face_detector.onnx",
    providers=["CPUExecutionProvider"]
)

input_name = session.get_inputs()[0].name
output_names = [o.name for o in session.get_outputs()]


In [ ]:
from flask import Flask, request, jsonify
import cv2
import numpy as np
import onnxruntime as ort

app = Flask(__name__)

# Cargar modelo
session = ort.InferenceSession(
    "face_detector.onnx",
    providers=["CPUExecutionProvider"]
)

input_name = session.get_inputs()[0].name
output_names = [o.name for o in session.get_outputs()]

def preprocess_image(image):
    """
    Ajusta esto según cómo fue entrenado tu modelo
    """
    # 1. Convertir a RGB (OpenCV carga en BGR)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # 2. Resize igual que en training
    image = cv2.resize(image, (224, 224))
    
    # 3. Normalización ImageNet
    image = image.astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    image = (image - mean) / std
    
    # 4. Transponer a CHW y añadir Batch dim
    image = np.transpose(image, (2, 0, 1))
    image = np.expand_dims(image, axis=0)
    return image.astype(np.float32)

@app.route("/detect", methods=["POST"])
def detect():
    if "image" not in request.files:
        return jsonify({"error": "No image provided"}), 400

    file = request.files["image"]
    image_bytes = np.frombuffer(file.read(), np.uint8)
    image = cv2.imdecode(image_bytes, cv2.IMREAD_COLOR)

    input_tensor = preprocess_image(image)

    outputs = session.run(
        output_names,
        {input_name: input_tensor}
    )

    # --- VISUALIZACIÓN ---
    score = outputs[0][0][0]
    bbox = outputs[1][0] # [x, y, w, h] normalizado
    
    detections = [float(score)]
    box_coords = []

    if score > 0.1:
        h_img, w_img, _ = image.shape
        x, y, w, h = bbox
        
        # Desnormalizar
        x_px = int(x * w_img)
        y_px = int(y * h_img)
        w_px = int(w * w_img)
        h_px = int(h * h_img)
        
        # Dibujar rectangulo (BGR para OpenCV)
        cv2.rectangle(image, (x_px, y_px), (x_px+w_px, y_px+h_px), (0, 0, 255), 2)
        cv2.putText(image, f"{score:.2f}", (x_px, y_px-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 2)
        
        box_coords = [x_px, y_px, w_px, h_px]
        
    # Guardar imagen resultante
    cv2.imwrite("detected.jpg", image)

    return jsonify({
        "detections": detections,
        "box": box_coords
    })

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.17.83.5:5000
Press CTRL+C to quit
127.0.0.1 - - [19/Dec/2025 09:03:25] "POST /detect HTTP/1.1" 200 -
127.0.0.1 - - [19/Dec/2025 09:03:45] "POST /detect HTTP/1.1" 200 -
127.0.0.1 - - [19/Dec/2025 09:04:20] "POST /detect HTTP/1.1" 200 -
127.0.0.1 - - [19/Dec/2025 09:04:43] "POST /detect HTTP/1.1" 200 -
127.0.0.1 - - [19/Dec/2025 09:05:17] "POST /detect HTTP/1.1" 200 -
127.0.0.1 - - [19/Dec/2025 09:06:06] "POST /detect HTTP/1.1" 200 -
